# 01 — Generate Tool-Calling Training Data (extensive)

**CPU only. Run in Modal notebook before notebook 02.**

Teaches Moshi, via its inner-monologue text stream, to:
1. Emit `<|tool_call|>get_time<|tool_end|>` when the user asks for the time
2. Emit `<|tool_call|>get_weather CITY<|tool_end|>` for weather
3. Read the injected `<|tool_result|>…<|tool_result_end|>` and speak a natural answer
4. **Re-call** the tool on follow-up questions instead of parroting a stale answer
5. **Stay quiet / not call** during silence and ordinary conversation

Design notes:
- **Call/result formats match runtime exactly** — call is `get_weather London`
  (plain city), result is `London: Clear, +27°C` (wttr.in `%C, %t`) and time is
  `8:31 PM on Friday, June 19, 2026` (`get_time()` output).
- **Masking is principled**: the model is trained (mask=1) only on the tokens it
  must *emit* — the tool call and the spoken response. The injected result block
  is context (mask=0) because at runtime it is force-fed, not predicted.
- **Heavy negatives**: silence, chit-chat, and distractor sentences that contain
  the words "time"/"weather" but must NOT trigger a tool.

Output saved to `data/tool_calls.jsonl`.

In [ ]:
# %pip installs into the *kernel's* env (unlike !pip which may hit another python)
%pip install sentencepiece huggingface_hub einops sphn numpy -q

In [ ]:
import json, random, sys, os
from pathlib import Path

# ── Config — fill these in ────────────────────────────────────────────────────
HF_PATCHED_REPO = "abrarfahim/moshi-tool-patched"
HF_TOKEN        = os.environ.get("HF_TOKEN", None)  # or paste token string here
# ─────────────────────────────────────────────────────────────────────────────

# Clone repo if not present (Modal environment)
REPO = Path("/repo")
if not REPO.exists():
    import subprocess
    subprocess.run(["git", "clone",
        "https://github.com/syedfahimabrar/moshi-D-gu.git",
        str(REPO)], check=True)

sys.path.insert(0, str(REPO / "moshi"))

import sentencepiece as spm

TOKENIZER_PATH = REPO / "weights/patched/tokenizer_spm_32k_3.model"
if not TOKENIZER_PATH.exists():
    from huggingface_hub import hf_hub_download
    TOKENIZER_PATH = Path(hf_hub_download(
        repo_id=HF_PATCHED_REPO, filename="tokenizer_spm_32k_3.model",
        local_dir="/tmp/patched", token=HF_TOKEN))

tok = spm.SentencePieceProcessor()
tok.Load(str(TOKENIZER_PATH))
print(f"Tokenizer: {tok.get_piece_size()} tokens")

In [ ]:
PAD_ID             = 3
TOOL_CALL_ID       = 32000
TOOL_END_ID        = 32001
TOOL_RESULT_ID     = 32002
TOOL_RESULT_END_ID = 32003

assert tok.id_to_piece(TOOL_CALL_ID) == "<|tool_call|>", "Tokenizer not patched!"
print("Special tokens:", [tok.id_to_piece(i) for i in [32000,32001,32002,32003]])

In [ ]:
# ── Core helpers ──────────────────────────────────────────────────────────────
def encode(text):
    """Encode text, dropping any special tokens (we add those explicitly)."""
    return [i for i in tok.encode(text) if i < 32000]

def build(segments, ex_type):
    """segments: list of (token_list, trainable_bool). Returns a dataset row.

    trainable=True  → mask=1 (the model must EMIT these: tool call + spoken reply)
    trainable=False → mask=0 (context only: user query, PAD gaps, injected result)
    """
    tokens, mask = [], []
    for toks, train in segments:
        tokens.extend(toks)
        mask.extend([1 if train else 0] * len(toks))
    return {"tokens": tokens, "mask": mask, "type": ex_type}

def gap(lo=4, hi=18):
    return [PAD_ID] * random.randint(lo, hi)

_DAYS   = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
_MONTHS = ["January","February","March","April","May","June",
           "July","August","September","October","November","December"]

# ── Time ───────────────────────────────────────────────────────────────────────
_TIME_TRIGGERS = [
    "what time is it", "what time is it now", "tell me the time",
    "what's the current time", "do you know what time it is",
    "can you check the time", "what time do you have",
    "I need to know the time", "got the time", "hey what's the time",
    "could you tell me the time", "what's the time right now",
    "do you have the time", "what time is it over there",
]
_TIME_FOLLOWUPS = [
    "what about now", "and now", "what time is it now",
    "can you check again", "and the time now", "how about now",
]
_TIME_RESPONSES = [
    "It's {s} right now.", "The time is {s}.", "Right now it's {s}.",
    "It's currently {s}.", "It's {s} at the moment.", "Looks like it's {s}.",
]

def _real_time():
    h, m = random.randint(1, 12), random.randint(0, 59)
    ap = random.choice(["AM", "PM"])
    day, mon, d = random.choice(_DAYS), random.choice(_MONTHS), random.randint(1, 28)
    result = f"{h}:{m:02d} {ap} on {day}, {mon} {d}, 2026"  # matches get_time()
    spoken = (f"{h} {ap.lower()}" if m == 0 else f"{h}:{m:02d} {ap.lower()}")
    return result, spoken

def _time_turn(trigger):
    result, spoken = _real_time()
    return [
        (encode(trigger), False),
        (gap(), False),
        ([TOOL_CALL_ID], True), (encode("get_time"), True), ([TOOL_END_ID], True),
        ([TOOL_RESULT_ID], False), (encode(result), False), ([TOOL_RESULT_END_ID], False),
        (encode(random.choice(_TIME_RESPONSES).format(s=spoken)), True),
    ]

def make_time_example():
    return build(_time_turn(random.choice(_TIME_TRIGGERS)), "time")

def make_time_multiturn():
    segs = _time_turn(random.choice(_TIME_TRIGGERS))
    segs.append((gap(8, 25), False))
    segs.extend(_time_turn(random.choice(_TIME_FOLLOWUPS)))
    return build(segs, "time_multiturn")

print("time:", tok.decode([t for t in make_time_example()["tokens"] if t < 32000]))

In [ ]:
# ── Weather ─────────────────────────────────────────────────────────────────────
_CITIES = [
    "London","New York","Tokyo","Paris","Sydney","Dubai","Mumbai","Toronto",
    "Berlin","Singapore","Los Angeles","Seoul","Bangkok","Istanbul","Cairo",
    "Amsterdam","Madrid","Rome","Hong Kong","Kuala Lumpur","Dhaka","Karachi",
    "Stockholm","Oslo","Helsinki","Moscow","Beijing","Delhi","Lagos","Nairobi",
    "Chicago","Boston","Seattle","Miami","Vancouver","Melbourne","Auckland",
]
# wttr.in %C condition strings (plain text, matches runtime get_weather)
_CONDITIONS = [
    "Clear", "Sunny", "Partly cloudy", "Cloudy", "Overcast", "Mist", "Fog",
    "Patchy rain nearby", "Light rain", "Moderate rain", "Heavy rain",
    "Light drizzle", "Thundery outbreaks", "Light snow", "Haze",
]
_WEATHER_RESPONSES = [
    "It's {cond} in {city} right now, about {temp}.",
    "Looks like {cond_l} in {city}, around {temp}.",
    "In {city} it's {cond_l}, {temp} out there.",
    "{city} is {cond_l} at the moment, {temp}.",
    "Right now {city} has {cond_l} skies, {temp}.",
]
_WEATHER_LOCAL_RESP = [
    "It's {cond_l} where you are, about {temp}.",
    "Locally it's {cond_l}, around {temp}.",
    "Right now it's {cond_l} outside, {temp}.",
]

def _real_weather(city):
    cond = random.choice(_CONDITIONS)
    t = random.randint(-8, 40)
    temp = f"+{t}°C" if t >= 0 else f"{t}°C"        # wttr.in %t style
    result = f"{city}: {cond}, {temp}"               # matches _fetch_one() output
    return result, cond, temp

def _weather_turn(trigger, city):
    result, cond, temp = _real_weather(city)
    resp = random.choice(_WEATHER_RESPONSES).format(
        city=city, cond=cond, cond_l=cond.lower(), temp=temp)
    return [
        (encode(trigger), False),
        (gap(), False),
        ([TOOL_CALL_ID], True), (encode(f"get_weather {city}"), True), ([TOOL_END_ID], True),
        ([TOOL_RESULT_ID], False), (encode(result), False), ([TOOL_RESULT_END_ID], False),
        (encode(resp), True),
    ]

def make_weather_example(city=None):
    city = city or random.choice(_CITIES)
    triggers = [f"what's the weather in {city}", f"weather in {city}",
                f"how's the weather in {city}", f"what's it like in {city}",
                f"is it raining in {city}", f"how hot is it in {city}",
                f"tell me the weather in {city}", f"check the weather in {city}"]
    return build(_weather_turn(random.choice(triggers), city), "weather")

def make_weather_local():
    """No city → model emits a bare get_weather (runtime fetches local)."""
    cond = random.choice(_CONDITIONS)
    t = random.randint(-8, 40)
    temp = f"+{t}°C" if t >= 0 else f"{t}°C"
    result = f"Local: {cond}, {temp}"
    trigger = random.choice([
        "what's the weather like", "how's the weather", "what's it like outside",
        "is it cold out", "do I need a jacket", "what's the weather today",
        "how's it looking outside", "is it raining",
    ])
    resp = random.choice(_WEATHER_LOCAL_RESP).format(cond_l=cond.lower(), temp=temp)
    segs = [
        (encode(trigger), False),
        (gap(), False),
        ([TOOL_CALL_ID], True), (encode("get_weather"), True), ([TOOL_END_ID], True),
        ([TOOL_RESULT_ID], False), (encode(result), False), ([TOOL_RESULT_END_ID], False),
        (encode(resp), True),
    ]
    return build(segs, "weather_local")

def make_weather_multiturn():
    c1, c2 = random.sample(_CITIES, 2)
    segs = _weather_turn(f"what's the weather in {c1}", c1)
    segs.append((gap(8, 25), False))
    segs.extend(_weather_turn(random.choice([f"and {c2}", f"what about {c2}",
                                             f"how about {c2}", f"and in {c2}"]), c2))
    return build(segs, "weather_multiturn")

print("weather:", tok.decode([t for t in make_weather_example("London")["tokens"] if t < 32000]))
print("local:  ", tok.decode([t for t in make_weather_local()["tokens"] if t < 32000]))

In [ ]:
# ── Negatives: teach the model NOT to call a tool ────────────────────────────────

# 1) Pure silence: only PADs. Train the tail to keep predicting PAD (stay quiet),
#    so a fresh session (just PADs) does NOT fire a tool call.
def make_silence_example():
    n = random.randint(20, 60)
    k = random.randint(n // 3, n // 2)   # first part context, rest trained as PAD
    tokens = [PAD_ID] * n
    mask   = [0] * k + [1] * (n - k)
    return {"tokens": tokens, "mask": mask, "type": "silence"}

# 2) Ordinary chit-chat: a question, then a normal spoken reply, no tool.
_CHITCHAT = [
    ("hey how are you doing", "I'm doing great, thanks for asking."),
    ("tell me a fun fact", "Sure — octopuses have three hearts."),
    ("what do you think about music", "I love it, music can really change a mood."),
    ("what's your favorite book", "I'm fond of a good science fiction novel."),
    ("can you recommend a movie", "If you like thrillers, Inception is a classic."),
    ("tell me about space", "Space is vast — billions of galaxies out there."),
    ("what is machine learning", "It's teaching computers to learn from data."),
    ("how do I cook pasta", "Boil salted water, add the pasta, simmer till tender."),
    ("what's a good hobby", "Photography is fun and gets you outdoors."),
    ("tell me a joke", "Why did the scarecrow win an award? He was outstanding in his field."),
    ("how was your day", "Pretty good, just enjoying the conversation."),
    ("what should I eat", "Maybe something light, like a fresh salad."),
]
def make_chitchat_example():
    q, a = random.choice(_CHITCHAT)
    segs = [(encode(q), False), (gap(), False), (encode(a), True)]
    return build(segs, "chitchat")

# 3) Distractors: contain the words "time"/"weather" but must NOT trigger a tool.
_DISTRACTORS = [
    ("I had a great time at the party", "Sounds like a lot of fun!"),
    ("long time no see, how have you been", "I've been well, good to catch up."),
    ("it's about time we got started", "Agreed, let's dive in."),
    ("the weather was lovely on our trip", "That makes for a perfect getaway."),
    ("I've been feeling under the weather", "Sorry to hear that, hope you feel better soon."),
    ("time flies when you're having fun", "It really does!"),
    ("we had a wonderful time in Italy", "Italy is beautiful this time of year."),
    ("once upon a time there was a castle", "Ooh, I love a good story."),
    ("that movie was a waste of time", "Shame when a film doesn't deliver."),
    ("let's weather the storm together", "Absolutely, we'll get through it."),
    ("do you have time for a quick chat", "Of course, I'm happy to talk."),
    ("the timing of that was perfect", "Couldn't have gone better."),
    ("she gave me a hard time about it", "That can be frustrating."),
    ("spring is my favorite time of year", "The blossoms make it special."),
]
def make_distractor_example():
    q, a = random.choice(_DISTRACTORS)
    segs = [(encode(q), False), (gap(), False), (encode(a), True)]
    return build(segs, "distractor")

print("silence  :", make_silence_example()["tokens"][:10], "...")
print("chitchat :", tok.decode([t for t in make_chitchat_example()["tokens"] if t < 32000]))
print("distract :", tok.decode([t for t in make_distractor_example()["tokens"] if t < 32000]))

In [ ]:
random.seed(42)
dataset = []

# ── Positives: tool calls ────────────────────────────────────────────────────
for _ in range(500):                       # time, single turn
    dataset.append(make_time_example())
for _ in range(150):                       # time, follow-up re-call
    dataset.append(make_time_multiturn())

for city in _CITIES:                       # guarantee coverage of every city
    for _ in range(12):
        dataset.append(make_weather_example(city))
for _ in range(150):                       # extra random single-city weather
    dataset.append(make_weather_example())
for _ in range(150):                       # local / no-city weather
    dataset.append(make_weather_local())
for _ in range(150):                       # weather follow-up re-call
    dataset.append(make_weather_multiturn())

# ── Negatives: do NOT call ────────────────────────────────────────────────────
for _ in range(200):                       # pure silence
    dataset.append(make_silence_example())
for _ in range(250):                       # ordinary chit-chat
    dataset.append(make_chitchat_example())
for _ in range(300):                       # time/weather words, no tool
    dataset.append(make_distractor_example())

random.shuffle(dataset)

counts = {}
for ex in dataset:
    counts[ex["type"]] = counts.get(ex["type"], 0) + 1

n_pos = sum(v for k, v in counts.items() if k.startswith(("time", "weather")))
n_neg = len(dataset) - n_pos
print(f"Total: {len(dataset)} examples  ({n_pos} positive / {n_neg} negative)")
print("Breakdown:", {k: counts[k] for k in sorted(counts)})
print(f"Avg length: {sum(len(e['tokens']) for e in dataset)/len(dataset):.0f} tokens")
print(f"Max length: {max(len(e['tokens']) for e in dataset)} tokens")

In [ ]:
OUT = REPO / "notebooks/data/tool_calls.jsonl"
OUT.parent.mkdir(parents=True, exist_ok=True)

with OUT.open("w") as f:
    for ex in dataset:
        f.write(json.dumps({"tokens": ex["tokens"], "mask": ex["mask"], "type": ex["type"]}) + "\n")

print(f"Saved {len(dataset)} examples → {OUT}")
print("\nNext: commit data/tool_calls.jsonl to git, then run notebook 02.")

In [ ]:
# Preview one of each type, marking which tokens are trained (mask=1, shown UPPER)
def render(ex):
    out = []
    for tid, m in zip(ex["tokens"], ex["mask"]):
        if   tid == TOOL_CALL_ID:       s = "‹CALL›"
        elif tid == TOOL_END_ID:        s = "‹END›"
        elif tid == TOOL_RESULT_ID:     s = "‹RES›"
        elif tid == TOOL_RESULT_END_ID: s = "‹/RES›"
        elif tid == PAD_ID:             s = "·"
        else:                           s = tok.id_to_piece(tid).replace("▁", " ")
        out.append(s if m else s.lower() if s.isalpha() else s)
    return "".join(out)

for t in ["time", "time_multiturn", "weather", "weather_local",
          "weather_multiturn", "silence", "chitchat", "distractor"]:
    ex = next((e for e in dataset if e["type"] == t), None)
    if ex:
        print(f"[{t}]")
        print("  ", render(ex)[:160])
        print(f"   trained tokens (mask=1): {sum(ex['mask'])}/{len(ex['mask'])}")
        print()